In [28]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict
import warnings
warnings.filterwarnings('ignore')



df = pd.read_csv('fundamentals_with_prices.csv')


df.isna().sum()[df.isna().sum() > 0]

df = df.dropna(subset=["close_price"])

df.isna().sum()[df.isna().sum() > 0]


df = df.sort_values(["ticker", "fiscalDateEnding"])

df["future_return_1y"] = ( df.groupby("ticker")["close_price"].shift(-4) / df["close_price"] - 1)



df = df.sort_values(["ticker","fiscalDateEnding"])

df["future_return_1y"] = (
    df.groupby("ticker")["close_price"].shift(-4) / df["close_price"] - 1
)

df = df.dropna(subset=["future_return_1y"])



df.groupby("ticker")["future_return_1y"].apply(lambda x: x.tail(4))



df["fiscalDateEnding"] = pd.to_datetime(df["fiscalDateEnding"])



df["quarter"] = df["fiscalDateEnding"].dt.to_period("Q")



quarters = sorted(df["quarter"].unique())

test_q = quarters[-1]      # latest quarter
val_q = quarters[-2]       # previous quarter

non_feature_cols = ["ticker", "fiscalDateEnding", 'close_price', 'future_return_1y', 'price_date', 'quarter']
target = "future_return_1y"

df[non_feature_cols] = ( df.groupby("quarter")[non_feature_cols].transform(lambda x: (x - x.mean()) / x.std()))

test_df = df[df["quarter"] == test_q]

val_df = df[df["quarter"] == val_q]

train_df = df[df["quarter"] < val_q]


df.isna().sum()[df.isna().sum() > 0]


print("Train quarters:", train_df["quarter"].unique())
print("Validation quarter:", val_q)
print("Test quarter:", test_q)


train_df.columns

non_feature_cols = ["ticker", "fiscalDateEnding", 'close_price', 'future_return_1y', 'price_date', 'quarter']
target = "future_return_1y"

X_train = train_df.drop(columns=non_feature_cols)
y_train = train_df[target]

X_val = val_df.drop(columns=non_feature_cols)
y_val = val_df[target]

X_test = test_df.drop(columns=non_feature_cols)
y_test = test_df[target]

#features = ["revenue_growth","profit_margin","roe","debt_to_equity"]  # your factors
#target = "future_return"
#X_train = train_df[features]
#y_train = train_df[target]
#X_val = val_df[features]
#y_val = val_df[target]



from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)



test_df["pred_return"] = model.predict(X_test)

top10 = test_df.sort_values("pred_return", ascending=False).head(10)

print(top10[["ticker","pred_return"]])



from scipy.stats import spearmanr

corr, _ = spearmanr(test_df["future_return_1y"], test_df["pred_return"])

print("Spearman Rank Correlation:", corr)



top10_each_q = ( test_df.sort_values(["fiscalDateEnding","pred_return"], ascending=[True, False]).groupby("fiscalDateEnding").head(10))



top10 = test_df.sort_values("pred_return", ascending=False).head(10)

print(top10[["ticker","pred_return","future_return_1y"]])


top10 = test_df.sort_values("pred_return", ascending=False).head(10)

top10["future_return_1y"].mean()


from lightgbm import LGBMRegressor, early_stopping

model = LGBMRegressor( n_estimators=300,
                        learning_rate=0.03,
                        max_depth=4,
                        num_leaves=20,
                        min_child_samples=50,
                        subsample=0.7,
                        colsample_bytree=0.7,
                        reg_alpha=1.0,
                        reg_lambda=1.0,
                        random_state=42
                    )


model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[early_stopping(50)]
)


test_df["pred_return"] = model.predict(X_test, num_iteration=model.best_iteration_)

top10 = test_df.sort_values("pred_return", ascending=False).head(10)

print(top10[["ticker","pred_return"]])



import pandas as pd

imp = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(imp.head(20))



TypeError: Could not convert string 'AAPLADBEADPAEPALGNAMDAMGNAMZNAZNBIDUBIIBCDNSCHTRDDOGFISVFTNTGILDGOOGLIDXXINTCISRGJDKHCKLACLCIDMETAMRNAMSFTMTCHNXPIPDDPYPLQCOMREGNSBUXSIRISWKS' to numeric